# 실험 결과 비교

In [1]:
import glob

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

RESULT_GLOB = "exp*/exp*.csv"
CLASS_NAMES = {"0": "Normal", "1": "Underfill_50FR", "2": "Underfill_Fan", "macro": "macro"}

result_files = sorted(glob.glob(RESULT_GLOB))
df_all = pd.concat([pd.read_csv(p).assign(source=p) for p in result_files], ignore_index=True)
df_all["class"] = df_all["class"].astype(str)
df_all = df_all.drop_duplicates(subset=["method", "seed", "class"], keep="last")

print("읽은 결과 파일:")
for p in result_files:
    print("  ", p)
print("\n실험별 seed 수:")
print(df_all[df_all["class"] == "macro"].groupby("method")["seed"].nunique().to_string())

읽은 결과 파일:
   exp1_baseline\exp1_baseline.csv
   exp2_oversampling_step_300\exp2_oversampling_step_300.csv
   exp3_joint_step_300\exp3_joint_step_300.csv

실험별 seed 수:
method
exp1_baseline                 10
exp2_oversampling_step_300    10
exp3_joint_step_300           10


## 1. 실험별 평균 ± 표준편차

In [2]:
METHOD_ORDER = sorted(df_all["method"].unique(), key=lambda m: (m.split("_")[0], m))


def per_seed(metric, cls="macro"):
    sub = df_all[df_all["class"] == str(cls)]
    return sub.pivot_table(index="seed", columns="method", values=metric, aggfunc="mean")[METHOD_ORDER]


summary_items = [("accuracy", "macro"), ("precision", "macro"), ("recall", "macro"), ("f1", "macro")]
summary = {}
for metric, cls in summary_items:
    tab = per_seed(metric, cls)
    name = f"{metric}" if cls == "macro" else f"{metric}_{CLASS_NAMES[cls]}"
    summary[name] = tab.mean().map("{:.3f}".format) + " ± " + tab.std(ddof=1).map("{:.3f}".format)
summary = pd.DataFrame(summary)
summary.insert(0, "n_seeds", per_seed("f1").notna().sum())
summary

,n_seeds,accuracy,precision,recall,f1
method,,,,,
exp1_baseline,10,0.512 ± 0.129,0.626 ± 0.121,0.512 ± 0.129,0.442 ± 0.178
exp2_oversampling_step_300,10,0.880 ± 0.052,0.887 ± 0.047,0.880 ± 0.052,0.880 ± 0.054
exp3_joint_step_300,10,0.779 ± 0.043,0.839 ± 0.022,0.779 ± 0.043,0.773 ± 0.045


## 2. paired t-test

In [3]:
COMPARISONS = [
    ("exp2_oversampling_step_300", "exp1_baseline", "오버샘플링 vs baseline"),
    ("exp3_joint_step_300", "exp1_baseline", "joint vs baseline"),
    ("exp3_joint_step_300", "exp2_oversampling_step_300", "joint vs 오버샘플링"),
]


def paired_tests(metric="f1", cls="macro", comparisons=COMPARISONS, df=None):
    df = df_all if df is None else df
    sub = df[df["class"] == str(cls)]
    table = sub.pivot_table(index="seed", columns="method", values=metric, aggfunc="mean")
    rows = []
    for a, b, desc in comparisons:
        if a not in table or b not in table:
            continue
        pair = table[[a, b]].dropna()
        n = len(pair)
        diff = pair[a] - pair[b]
        row = {"비교": desc, "A": a, "B": b, "n": n,
               "mean_A": pair[a].mean(), "mean_B": pair[b].mean()}
        if n >= 2 and diff.std(ddof=1) > 0:
            t_stat, p_val = stats.ttest_rel(pair[a], pair[b])
            row.update({"t": t_stat, "p": p_val})
        else:
            row.update({"t": np.nan, "p": np.nan})
        rows.append(row)
    res = pd.DataFrame(rows)
    if res.empty:
        print(f"[{metric}/{cls}] 비교할 수 있는 실험 쌍이 아직 없습니다.")
        return res
    res["유의(p<0.05)"] = res["p"] < 0.05
    return res


pd.set_option("display.float_format", "{:.4f}".format)
paired_tests("f1", "macro")


,비교,A,B,n,mean_A,mean_B,t,p,유의(p<0.05)
0,오버샘플링 vs baseline,exp2_oversampling_step_300,exp1_baseline,10,0.8796,0.4423,9.5317,0.0000,True
1,joint vs baseline,exp3_joint_step_300,exp1_baseline,10,0.7728,0.4423,5.9365,0.0002,True
2,joint vs 오버샘플링,exp3_joint_step_300,exp2_oversampling_step_300,10,0.7728,0.8796,-5.3702,0.0005,True
